In [6]:
import warnings
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf # for OLS (smf.ols)
from sklearn.metrics import mean_squared_error, mean_absolute_error # perform mean-squared for RMSE and MAE
from statsmodels.stats.anova import anova_lm # perform Chi-squared test


# IGNORE WARNINGS
warnings.filterwarnings("ignore")

# LOAD DATA

# 1. base data
train = pd.read_csv("data/GDP_Forecast_train.csv")
test  = pd.read_csv("data/GDP_Forecast_test.csv")
gdp_pca = pd.read_csv("data/GDP_PCA_for_test.csv") # validation data for test

# 2. non-financial external indicators
houst = pd.read_csv("data/HOUST_PCA.csv")
indpro = pd.read_csv("data/INDPRO_PCA.csv")
unrate = pd.read_csv("data/UNRATE_PCA.csv")

# 3. financial external indicators
compustat = pd.read_csv("data/compustat_quarterly_financials2.csv")




In [7]:
## CONVERT YEAR-QUARTER: SPLIT COLUMNS

# ensure year is an integer
train["year"] = train["YQ"].str[:4].astype(int)
test["year"] = test["YQ"].str[:4].astype(int)

# keep qtr as string / category
train["qtr"] = train["YQ"].str[4:]
test["qtr"] = test["YQ"].str[4:]

In [8]:
## USING SEPARATE gdp_pca DATASET FOR gdp_pcaING RMSE
# we are using a different dataset for gdp_pca because we want to calculate RMSE before submitting

# convert the data to YQ, year and quarter
gdp_pca["observation_date"] = pd.to_datetime(gdp_pca["observation_date"], dayfirst = True)
gdp_pca["year"] = gdp_pca["observation_date"].dt.year
gdp_pca["qtr"] = "Q" + gdp_pca["observation_date"].dt.quarter.astype(str)
gdp_pca["YQ"] = gdp_pca["year"].astype(str) + gdp_pca["qtr"]

# remove observation_date column and move GDP to the back while changing col name to NGDP
gdp_pca.drop(columns = "observation_date", inplace = True)
col = gdp_pca.pop("GDP_PCA")
gdp_pca["NGDP"] = col

# get the quarterly
gdp_pca.sort_values(by = "YQ")

# # check data for 1990 - 2015 is similar to train to continue with this dataset for gdp_pca
# gdp_pca[gdp_pca["year"] <= 2015].transpose()
## NOTE: data from later years is not as accurate, but we will use this to tentatively gdp_pca our RMSE
gdp_pca = gdp_pca[gdp_pca["year"]>= 2016].reset_index(drop = True)

## PREVIEWING OF GDP_PCA (VALIDATION ONLY)

print("GDP_PCA preview:")
print(gdp_pca.head(8))
print(gdp_pca.tail(8))
print("\nGDP_PCA info:")
print("Number of rows:", len(gdp_pca))
print("Number of unique quarters:", gdp_pca["YQ"].nunique())
print("Any missing YQ?", gdp_pca["YQ"].isna().any())
print("Any missing GDP_PCA?", gdp_pca["NGDP"].isna().any())
print("\nCheck duplicate quarters in GDP_PCA:", gdp_pca["YQ"].duplicated().sum())

GDP_PCA preview:
   year qtr      YQ  NGDP
0  2016  Q1  2016Q1   2.0
1  2016  Q2  2016Q2   4.1
2  2016  Q3  2016Q3   3.9
3  2016  Q4  2016Q4   4.2
4  2017  Q1  2017Q1   4.1
5  2017  Q2  2017Q2   3.3
6  2017  Q3  2017Q3   5.3
7  2017  Q4  2017Q4   7.2
    year qtr      YQ  NGDP
12  2019  Q1  2019Q1   3.8
13  2019  Q2  2019Q2   5.5
14  2019  Q3  2019Q3   6.1
15  2019  Q4  2019Q4   4.0
16  2020  Q1  2020Q1  -3.3
17  2020  Q2  2020Q2 -29.1
18  2020  Q3  2020Q3  39.9
19  2020  Q4  2020Q4   7.2

GDP_PCA info:
Number of rows: 20
Number of unique quarters: 20
Any missing YQ? False
Any missing GDP_PCA? False

Check duplicate quarters in GDP_PCA: 0


In [9]:
## DEFINE FUNCTIONS FOR REUSE

def calc_performance(formula1, model1, formula2, model2):
    print(pd.DataFrame({'model'   : [formula1, formula2],
                    'Adj.R^2' : [model1.rsquared_adj, model2.rsquared_adj]}))
    print()
    print(anova_lm(model1, model2, test = "Chisq"))

def create_submission(test_df, y_pred, filename):
    df = pd.DataFrame({"YQ": test_df["YQ"], "NGDP": y_pred})
    df.to_csv(f"attempts/{filename}.csv", index = False)

Base Model

In [11]:
## CONVERT TO COMMON QUARTER KEY

train["quarter"] = pd.PeriodIndex(train["YQ"], freq = "Q")
test["quarter"]  = pd.PeriodIndex(test["YQ"],  freq = "Q")


## CREATE LAG VARIABLE

# sort training data by quarter before creating lag
train = train.sort_values(by = "quarter").reset_index(drop = True)

# create 1-quarter lag of NGDP
train["NGDP_lag1"] = train["NGDP"].shift(1)

# remove first row with missing lag
train_model = train.dropna().copy()


## FIT BASE MODEL
# NGDP_t = a + b * NGDP_(t-1)

import statsmodels.api as sm

X = sm.add_constant(train_model["NGDP_lag1"])
y = train_model["NGDP"]

model_base = sm.OLS(y, X).fit()

print(model_base.summary())


                            OLS Regression Results                            
Dep. Variable:                   NGDP   R-squared:                       0.192
Model:                            OLS   Adj. R-squared:                  0.184
Method:                 Least Squares   F-statistic:                     24.00
Date:                Thu, 12 Mar 2026   Prob (F-statistic):           3.68e-06
Time:                        21:10:08   Log-Likelihood:                -237.24
No. Observations:                 103   AIC:                             478.5
Df Residuals:                     101   BIC:                             483.7
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.5247      0.479      5.273      0.0

In [12]:
## RECURSIVE FORECASTING USING BASE MODEL
# use last observed NGDP from training set as the starting lag

test = test.sort_values(by = "quarter").reset_index(drop = True)

base_forecast = []

last_ngdp = train["NGDP"].iloc[-1]

for i in range(len(test)):
    
    X_new = pd.DataFrame({
        "const"     : [1],
        "NGDP_lag1" : [last_ngdp]
    })
    
    pred = model_base.predict(X_new)[0]
    
    base_forecast.append(pred)
    
    # update lag for next quarter
    last_ngdp = pred

test["base_forecast"] = base_forecast


## PREVIEW FORECAST RESULTS

print("Base forecast preview:")
print(test[["YQ", "base_forecast"]].head(8))
print(test[["YQ", "base_forecast"]].tail(8))

Base forecast preview:
       YQ  base_forecast
0  2016Q1       2.743384
1  2016Q2       3.724569
2  2016Q3       4.153708
3  2016Q4       4.341400
4  2017Q1       4.423491
5  2017Q2       4.459395
6  2017Q3       4.475098
7  2017Q4       4.481966
        YQ  base_forecast
12  2019Q1       4.487220
13  2019Q2       4.487268
14  2019Q3       4.487289
15  2019Q4       4.487298
16  2020Q1       4.487302
17  2020Q2       4.487304
18  2020Q3       4.487304
19  2020Q4       4.487305


In [14]:
## VALIDATE BASE MODEL FORECAST AGAINST GDP_PCA

# keep only the predicted values needed for validation
base_validation = test[["YQ", "base_forecast"]].merge(
    gdp_pca[["YQ", "NGDP"]],
    on = "YQ",
    how = "left"
)

# rename columns clearly
base_validation.rename(columns = {
    "base_forecast" : "NGDP_pred",
    "NGDP"          : "NGDP_actual"
}, inplace = True)


## CALCULATE RMSE AND MAE

rmse_base = np.sqrt(mean_squared_error(
    base_validation["NGDP_actual"],
    base_validation["NGDP_pred"]
))

mae_base = mean_absolute_error(
    base_validation["NGDP_actual"],
    base_validation["NGDP_pred"]
)

print("Base model RMSE:", round(rmse_base, 4))
print("Base model MAE :", round(mae_base, 4))


## PREVIEW VALIDATION TABLE

print("\nBase validation preview:")
print(base_validation.head(8))
print(base_validation.tail(8))
print("\nAny missing actual values?", base_validation["NGDP_actual"].isna().any())
print("Any missing predicted values?", base_validation["NGDP_pred"].isna().any())

Base model RMSE: 11.1175
Base model MAE : 4.7121

Base validation preview:
       YQ  NGDP_pred  NGDP_actual
0  2016Q1   2.743384          2.0
1  2016Q2   3.724569          4.1
2  2016Q3   4.153708          3.9
3  2016Q4   4.341400          4.2
4  2017Q1   4.423491          4.1
5  2017Q2   4.459395          3.3
6  2017Q3   4.475098          5.3
7  2017Q4   4.481966          7.2
        YQ  NGDP_pred  NGDP_actual
12  2019Q1   4.487220          3.8
13  2019Q2   4.487268          5.5
14  2019Q3   4.487289          6.1
15  2019Q4   4.487298          4.0
16  2020Q1   4.487302         -3.3
17  2020Q2   4.487304        -29.1
18  2020Q3   4.487304         39.9
19  2020Q4   4.487305          7.2

Any missing actual values? False
Any missing predicted values? False


Model with non-accounting macro data

In [7]:
# Load test set and external indicators

houst = pd.read_csv("data/HOUST_PCA.csv")
indpro = pd.read_csv("data/INDPRO_PCA.csv")
unrate = pd.read_csv("data/UNRATE_PCA.csv")

# convert to quarter
houst["observation_date"] = pd.to_datetime(houst["observation_date"], format="mixed", dayfirst=True)
indpro["observation_date"] = pd.to_datetime(indpro["observation_date"], format="mixed", dayfirst=True)
unrate["observation_date"] = pd.to_datetime(unrate["observation_date"], format="mixed", dayfirst=True)

houst["quarter"] = pd.PeriodIndex(houst["observation_date"], freq="Q")
indpro["quarter"] = pd.PeriodIndex(indpro["observation_date"], freq="Q")
unrate["quarter"] = pd.PeriodIndex(unrate["observation_date"], freq="Q")

# keep needed columns only
houst = houst[["quarter", "HOUST_PCA"]].sort_values("quarter").reset_index(drop=True)
indpro = indpro[["quarter", "INDPRO_PCA"]].sort_values("quarter").reset_index(drop=True)
unrate = unrate[["quarter", "UNRATE_PCA"]].sort_values("quarter").reset_index(drop=True)

# create lag variable
houst["HOUST_lag1"] = houst["HOUST_PCA"].shift(1)
indpro["INDPRO_lag1"] = indpro["INDPRO_PCA"].shift(1)
unrate["UNRATE_lag1"] = unrate["UNRATE_PCA"].shift(1)

# merge lag variable
train_ext = train.copy()

train_ext = train_ext.merge(
    houst[["quarter","HOUST_lag1"]],
    on="quarter",
    how="left"
)

train_ext = train_ext.merge(
    indpro[["quarter","INDPRO_lag1"]],
    on="quarter",
    how="left"
)

train_ext = train_ext.merge(
    unrate[["quarter","UNRATE_lag1"]],
    on="quarter",
    how="left"
)

